# 6-1 계산 그래프와 Chain Rule — 심화

직접 작성한 코드와 저장된 실행 결과를 정리했습니다. 두 번째 심화 셀은 문법 오류를 바로잡았으며 재실행이 필요합니다.


In [ ]:
# 검증 가능 정답 코드
# 수기 감사에서는 loss→score의 local gradient만 구한 뒤 score→weight의 입력 계수를 일부러 누락해 장애를 재현합니다.
import torch

w = torch.tensor(1.5, requires_grad=True)
x = torch.tensor(2.0)
b = torch.tensor(-0.5)
target = torch.tensor(1.0)

# forward 계산을 중간 변수로 남겨야 어느 local gradient가 빠졌는지 추적하기 쉽습니다.
score = w * x + b
loss = (score - target) ** 2

# 장애 문서는 d(score)/dw=x를 빠뜨렸습니다.
buggy_grad = 2 * (score - target)
manual_grad = 2 * (score - target) * x

# backward는 Tensor loss에 호출합니다. item()으로 바꾼 값에는 그래프가 없습니다.
loss.backward()

# 같은 score와 loss에서 Autograd가 준 leaf gradient와 비교해 빠진 chain-rule 계수 x를 숫자로 확인합니다.
print(f"score={score.item():.2f}")
print(f"loss={loss.item():.2f}")
print(f"buggy_grad={buggy_grad.item():.2f}")
print(f"autograd_grad={w.grad.item():.2f}")
print("root_cause=dscore/dw=x was omitted")

score=2.50
loss=2.25
buggy_grad=3.00
autograd_grad=6.00
root_cause=dscore/dw=x was omitted


In [ ]:
# 검증 가능 정답 코드
# 공유 중간값 u를 한 번 계산하고 a·b가 서로 다른 계수로 기여하도록 두 출력 경로를 구성합니다.
import torch

a = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)

# u를 따로 두면 공통 경로와 파라미터별 경로가 눈에 보입니다.
u = 2 * a + b
risk = u ** 2

manual_a = 2 * u * 2  # d(risk)/du * du/da
manual_b = 2 * u * 1  # d(risk)/du * du/db
risk.backward()

match = torch.allclose(a.grad, manual_a) and torch.allclose(b.grad, manual_b)
# Autograd 결과를 손계산 2u×분기 계수와 대조해 공유 그래프의 각 leaf 기여가 맞는지 검산합니다.
print(f"u={u.item():.1f}")
print(f"grad_a={a.grad.item():.1f}")
print(f"grad_b={b.grad.item():.1f}")
print(f"manual_match={match}")

In [3]:
# 검증 가능 정답 코드
# 같은 leaf가 두 경로에 쓰이므로 경로를 분리한 그래프에서도 동일 입력과 weight를 유지해 기여를 비교합니다.
import torch

w = torch.tensor(1.0, requires_grad=True)
x1 = torch.tensor(2.0)
x2 = torch.tensor(3.0)

loss1 = (w * x1) ** 2
loss2 = (w * x2) ** 2
total_loss = loss1 + loss2

# 각 경로의 수기 기여입니다. 동일한 w를 공유하므로 최종 .grad에는 합이 저장됩니다.
path1_grad = 2 * w.detach() * x1 ** 2
path2_grad = 2 * w.detach() * x2 ** 2
total_loss.backward()

# 두 경로 gradient의 합과 전체 그래프 leaf gradient를 대조한 뒤 더 큰 path2를 첫 조사 대상으로 정합니다.
print(f"path1_grad={path1_grad.item():.1f}")
print(f"path2_grad={path2_grad.item():.1f}")
print(f"total_grad={w.grad.item():.1f}")
print("first_investigation=path2")

path1_grad=8.0
path2_grad=18.0
total_grad=26.0
first_investigation=path2
